# Demo — Kafka Consumer

Reads events that the producer demo wrote. Run [`demo_produce.ipynb`](./demo_produce.ipynb) first, otherwise there is nothing to read.

## 1. Configure the consumer

A consumer needs three settings:

- **`bootstrap.servers`** — same as for the producer.
- **`group.id`** — every consumer belongs to a *consumer group*. Kafka   remembers "how far" each *group* has read by storing one offset per   group per partition. Two consumers in the same group split the work;   two consumers in different groups read everything independently.
- **`auto.offset.reset`** — what to do when the group has no stored   position yet. `earliest` = read from the beginning, `latest` = read   only new messages.

In [ ]:
from confluent_kafka import Consumer

BROKER = 'redpanda:29092'

consumer = Consumer({
    'bootstrap.servers': BROKER,
    'group.id':          'demo-consumer',
    'auto.offset.reset': 'earliest',
})
consumer.subscribe(['strom'])
print('Consumer subscribed to: strom')

## 2. Poll for messages

`consumer.poll(timeout)` does several things:

1. Asks the broker for the next batch of messages.
2. Returns one message at a time, or `None` if nothing arrived within    `timeout` seconds.
3. Updates the in-memory offset position.

We loop until we have read 5 messages or hit 3 empty polls (~6 s with the 2-s timeout). Long-running consumers usually loop forever.

In [ ]:
received    = 0
empty_polls = 0

while received < 5 and empty_polls < 3:
    msg = consumer.poll(timeout=2.0)
    if msg is None:           # no message arrived within 2 s
        empty_polls += 1
        continue
    if msg.error():           # broker reported a transient error
        print(f'Error: {msg.error()}')
        continue

    empty_polls = 0
    received   += 1
    key   = msg.key().decode()   if msg.key()   else 'None'
    value = msg.value().decode() if msg.value() else 'None'
    print(f'[{received}] partition={msg.partition()} '
          f'offset={msg.offset()} key={key}')
    print(f'     value={value}')

consumer.close()              # releases the partition assignment
print(f'Done — {received} event(s) received.')

## What you should see

Each printed line shows the message's address (partition + offset), the routing key, and the JSON payload. **Run the producer demo again** and re-run this notebook: the offset increases — Kafka remembers what your group already read and gives you only the new ones.